# SCRESIA thesis-decision ladder: final Colab runner

This is the paper-facing runner for the Garrido thesis-decision ladder. It treats the thesis variables as faithful, but declares the protocol relaxations explicitly:

- L0: original Garrido static designs.
- L1a: common inventory period crossed with shift capacity, `thesis_factorized` / `MultiDiscrete([6, 3])`.
- L1b: per-node inventory period crossed with shift capacity, `factorized + per_node` / `MultiDiscrete([6, 6, 6, 3])`.
- E2a/E2b: PPO adaptivity, compared against the best static policy in the same action space.

Default profile runs the practical L1a job. Change `RUN_PROFILE` to `smoke`, `overnight_l1b`, `ppo_e2a_debug`, or `ppo_e2a_serious` as needed.


In [ ]:
# =====================
# 0) Run config
# =====================

RUN_PROFILE = "l1a_today"  # smoke, l1a_today, overnight_l1b, ppo_e2a_debug, ppo_e2a_serious

GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"
REPO_DIR_COLAB = "/content/scresia"
REPO_DIR_KAGGLE = "/kaggle/working/scresia"

USE_GOOGLE_DRIVE_OUTPUT = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/scresia_results/thesis_decision_ladder_final"
LOCAL_OUTPUT_DIR = "/content/thesis_decision_ladder_final"
KAGGLE_OUTPUT_DIR = "/kaggle/working/thesis_decision_ladder_final"

RUNTIME_PIP_PACKAGES = [
    "simpy>=4.1",
    "numpy>=1.26",
    "gymnasium>=0.29",
    "stable-baselines3>=2.3",
    "sb3-contrib>=2.3",
    "pandas>=2.2",
]

BASE_SEED = 42
REWARD_MODE = "ReT_cd_v1"
RISK_LEVEL = "increased"
OBSERVATION_VERSION = "v5"
OBSERVATION_MODE = "env_sdm_history_reward"
STEP_SIZE_HOURS = 168.0
STOCHASTIC_PT = True
GARRIDO_CFIS = "31-90"

PROFILE_CONFIG = {
    "smoke": dict(
        run_static=True,
        levels=["L0_garrido", "L1a_uniform_IxS"],
        replications=1,
        garrido_cfis="31",
        max_steps=4,
        run_ppo_e2a=False,
        run_ppo_e2b=False,
        ppo_seeds=[101],
        ppo_timesteps=256,
        ppo_eval_episodes=1,
    ),
    "l1a_today": dict(
        run_static=True,
        levels=["L0_garrido", "L1a_uniform_IxS"],
        replications=30,
        garrido_cfis=GARRIDO_CFIS,
        max_steps=260,
        run_ppo_e2a=False,
        run_ppo_e2b=False,
        ppo_seeds=[101, 202, 303],
        ppo_timesteps=200_000,
        ppo_eval_episodes=5,
    ),
    "overnight_l1b": dict(
        run_static=True,
        levels=["L1b_per_node_IxS"],
        replications=30,
        garrido_cfis=GARRIDO_CFIS,
        max_steps=260,
        l1b_screening_replications=5,
        l1b_top_k=20,
        run_ppo_e2a=False,
        run_ppo_e2b=False,
        ppo_seeds=[101, 202, 303],
        ppo_timesteps=200_000,
        ppo_eval_episodes=5,
    ),
    "ppo_e2a_debug": dict(
        run_static=False,
        levels=[],
        replications=1,
        garrido_cfis="31",
        max_steps=8,
        run_ppo_e2a=True,
        run_ppo_e2b=False,
        ppo_seeds=[101],
        ppo_timesteps=512,
        ppo_eval_episodes=1,
    ),
    "ppo_e2a_serious": dict(
        run_static=False,
        levels=[],
        replications=30,
        garrido_cfis=GARRIDO_CFIS,
        max_steps=260,
        run_ppo_e2a=True,
        run_ppo_e2b=False,
        ppo_seeds=[101, 202, 303],
        ppo_timesteps=200_000,
        ppo_eval_episodes=5,
    ),
}

cfg = PROFILE_CONFIG[RUN_PROFILE]
print({"RUN_PROFILE": RUN_PROFILE, **cfg})


In [ ]:
# ==========================================
# 1) Portable setup: Colab, Kaggle, or local
# ==========================================

from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").exists()
if not (IN_COLAB or IN_KAGGLE):
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {cmd}")
    return result


def install_runtime_packages() -> None:
    if IN_COLAB or IN_KAGGLE:
        run_cmd([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PIP_PACKAGES])
    else:
        print("local run: skipping pip install")


def clone_repo(target: str) -> Path:
    target_path = Path(target)
    shutil.rmtree(target_path, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(target_path)])
    return target_path


drive_mounted = False
if IN_COLAB and USE_GOOGLE_DRIVE_OUTPUT:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True, timeout_ms=120_000)
        drive_mounted = True
    except Exception as exc:
        print("WARNING: Drive mount failed; using local /content outputs.", repr(exc))

if IN_COLAB:
    repo_root = clone_repo(REPO_DIR_COLAB)
    output_root = Path(DRIVE_OUTPUT_DIR if drive_mounted else LOCAL_OUTPUT_DIR)
elif IN_KAGGLE:
    repo_root = clone_repo(REPO_DIR_KAGGLE)
    output_root = Path(KAGGLE_OUTPUT_DIR)
else:
    cwd = Path.cwd().resolve()
    repo_root = cwd if (cwd / "supply_chain").exists() else cwd.parent
    output_root = repo_root / "outputs" / "thesis_decision_ladder_final"

install_runtime_packages()
sys.path.insert(0, str(repo_root.parent))
sys.path.insert(0, str(repo_root))
output_root.mkdir(parents=True, exist_ok=True)

print("repo_root:", repo_root)
print("output_root:", output_root)
print("python:", sys.version)
print("platform:", platform.platform())
run_cmd(["git", "rev-parse", "--short", "HEAD"], cwd=repo_root, check=False)


In [ ]:
# ======================================
# 2) Verify the required repo contracts
# ======================================

import pandas as pd

ppo_help = run_cmd([sys.executable, "scripts/run_thesis_decision_ppo_smoke.py", "--help"], cwd=repo_root).stdout
ladder_help = run_cmd([sys.executable, "scripts/run_thesis_decision_ladder.py", "--help"], cwd=repo_root).stdout
assert "thesis_factorized" in ppo_help, "PPO smoke script lacks thesis_factorized mode."
assert "--inventory-period-mode" in ppo_help, "PPO smoke script lacks inventory-period-mode."
assert "L1b_per_node_IxS" in ladder_help, "Static ladder script lacks L1b per-node level."
print("contract check: ok")


In [ ]:
# ==============================
# 3) Static ladder: L0/L1a/L1b
# ==============================

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
static_dir = output_root / "static_ladder"
static_label = f"{RUN_PROFILE}_{run_stamp}"

if cfg["run_static"]:
    static_cmd = [
        sys.executable,
        "scripts/run_thesis_decision_ladder.py",
        "--label", static_label,
        "--output-root", str(static_dir),
        "--levels", *cfg["levels"],
        "--garrido-cfis", cfg["garrido_cfis"],
        "--replications", str(cfg["replications"]),
        "--reward-mode", REWARD_MODE,
        "--risk-level", RISK_LEVEL,
        "--observation-version", OBSERVATION_VERSION,
        "--observation-mode", OBSERVATION_MODE,
        "--step-size-hours", str(STEP_SIZE_HOURS),
        "--max-steps", str(cfg["max_steps"]),
    ]
    if STOCHASTIC_PT:
        static_cmd.append("--stochastic-pt")
    if "l1b_screening_replications" in cfg:
        static_cmd += ["--l1b-screening-replications", str(cfg["l1b_screening_replications"])]
    if "l1b_top_k" in cfg:
        static_cmd += ["--l1b-top-k", str(cfg["l1b_top_k"])]
    run_cmd(static_cmd, cwd=repo_root)
    static_run_dir = static_dir / static_label
else:
    static_run_dir = None
    print("static ladder skipped by profile")


In [ ]:
# =================================
# 4) PPO adaptivity: E2a and E2b
# =================================

ppo_dir = output_root / "ppo_adaptive"
ppo_run_dirs = []


def run_ppo(label_prefix: str, *, action_space_mode: str, inventory_period_mode: str):
    for seed in cfg["ppo_seeds"]:
        label = f"{label_prefix}_seed_{seed}_{run_stamp}"
        cmd = [
            sys.executable,
            "scripts/run_thesis_decision_ppo_smoke.py",
            "--label", label,
            "--output-root", str(ppo_dir),
            "--algo", "ppo_mlp",
            "--action-space-mode", action_space_mode,
            "--inventory-period-mode", inventory_period_mode,
            "--observation-version", OBSERVATION_VERSION,
            "--observation-mode", OBSERVATION_MODE,
            "--reward-mode", REWARD_MODE,
            "--risk-level", RISK_LEVEL,
            "--step-size-hours", str(STEP_SIZE_HOURS),
            "--max-steps", str(cfg["max_steps"]),
            "--train-timesteps", str(cfg["ppo_timesteps"]),
            "--eval-episodes", str(cfg["ppo_eval_episodes"]),
            "--seed", str(seed),
            "--garrido-cfis", cfg["garrido_cfis"],
            "--include-static-grid",
            "--eval-ai-on-garrido-cfis",
            "--learn-initial-decision",
            "--policy-net-arch", "medium",
        ]
        if STOCHASTIC_PT:
            cmd.append("--stochastic-pt")
        run_cmd(cmd, cwd=repo_root)
        ppo_run_dirs.append(ppo_dir / label)

if cfg["run_ppo_e2a"]:
    run_ppo("E2a_ppo_mlp_thesis_factorized", action_space_mode="thesis_factorized", inventory_period_mode="thesis_strict")
else:
    print("E2a PPO skipped by profile")

if cfg["run_ppo_e2b"]:
    run_ppo("E2b_ppo_mlp_per_node", action_space_mode="factorized", inventory_period_mode="per_node")
else:
    print("E2b PPO skipped by profile")


In [ ]:
# ===========================
# 5) Read result summaries
# ===========================

summary_rows = []

if static_run_dir is not None:
    static_summary_path = static_run_dir / "summary.json"
    static_policy_path = static_run_dir / "policy_summary.csv"
    if static_summary_path.exists():
        static_summary = json.loads(static_summary_path.read_text())
        print("static best policy:")
        print(json.dumps(static_summary.get("best_policy_by_fill_rate"), indent=2))
    if static_policy_path.exists():
        static_policy_summary = pd.read_csv(static_policy_path)
        display(static_policy_summary.sort_values("fill_rate_order_level_mean", ascending=False).head(20))

for run_dir in ppo_run_dirs:
    summary_path = run_dir / "summary.json"
    if summary_path.exists():
        payload = json.loads(summary_path.read_text())
        best_grid = payload.get("best_static_grid_by_fill_rate")
        promotion_gate = payload.get("promotion_gate", {})
        summary_rows.append({
            "run": run_dir.name,
            "action_space_mode": payload.get("action_space_mode"),
            "inventory_period_mode": payload.get("inventory_period_mode") or payload.get("env_kwargs", {}).get("inventory_period_mode"),
            "train_timesteps": payload.get("train_timesteps"),
            "best_static_grid": None if best_grid is None else best_grid.get("policy"),
            "beats_best_garrido_fill_rate": promotion_gate.get("beats_best_garrido_fill_rate"),
        })

if summary_rows:
    display(pd.DataFrame(summary_rows))
else:
    print("No PPO summaries for this profile.")


## Claim boundary

Use these outputs to argue sequential relaxation, not a single undifferentiated claim that PPO beats Garrido. The clean comparison is adaptive PPO against the best static policy inside the same action space. `factorized + per_node` is a declared extension; `thesis_factorized + thesis_strict` is the common-period thesis-decision lane.
